In [2]:
%reload_ext autoreload
%autoreload 2


from stnd.utility.data_utils import make_or_load_from_cache
from stnd.utility.utils import apply_random_seed
import sys
import os
import sklearn
import numpy as np
import pandas as pd
from datasets import load_dataset
import pickle

ROOT_PATH = os.path.join(os.path.dirname(os.path.abspath("")))
sys.path.insert(0, ROOT_PATH)
from experiments import (
    RANDOM_SEED,
    make_train_test_model_embeddings,
    make_cache_subpath,
    make_disagreement_scores_dict,
    make_fitted_weights
)
from utils import (
    lb_scenarios,
    dump_pickle,
    load_pickle,
    prepare_and_split_data
)
from plots import (
    MODEL_OUTPUTS_PATH,
    load_scores,
    safe_spearmanr
)
from selection import sample_items
from run_experiment import load_and_split_model_outputs
from acc import (
    compute_true_acc,
    compute_acc_knn
)
from utils_for_notebooks import read_per_model_info
from scripts.evaluate_mmlu import evaluate_mmlu
sys.path.pop(0);


CACHE_DIR = "./cache_dir"

In [5]:


# Path to the model outputs pickle file
model_outputs_pickle_path = "/weka/oh/arubinstein17/github/disco-public/data/model_outputs.pickle"

# Load the model outputs
with open(model_outputs_pickle_path, "rb") as f:
    model_outputs = pickle.load(f)

df = pd.read_csv(os.path.join(ROOT_PATH, "benchmark_csvs","open-llm-leaderboard.csv"))

# Optionally, print keys for inspection
print("Loaded model_outputs.pickle keys:", model_outputs.keys())
# Map model names in model_outputs["models"] to their row in leaderboard
model_names_from_outputs = model_outputs["models"]

# Build a mapping from lower-case model name in df['Model'] to their average score, to avoid case mismatches
# Some preprocessing to help with lookup due to potential variations in model naming
df["Model_lc"] = df["Model"].str.strip().str.lower()
df_model_to_avg = dict(zip(df["Model_lc"], df["Average ⬆️"]))

# Try to match model_outputs names (may need adjustment based on actual string matching)
avg_scores = []
fixed_model_names = []
for m in model_names_from_outputs:
    # If the model name is of the form 'open-llm-leaderboard/details_zhengr__MixTAO-7Bx2-MoE-DPO',
    # convert to 'zhengr/MixTAO-7Bx2-MoE-DPO' before matching.
    if isinstance(m, str) and m.startswith("open-llm-leaderboard/"):
        post_details = m.split("open-llm-leaderboard/details_")[1]
        creator, model = post_details.split("__")
        m_fixed = f"{creator}/{model}"
    else:
        m_fixed = m
    m_lc = str(m_fixed).strip().lower()
    avg_score = df_model_to_avg.get(m_lc, None)
    avg_scores.append(avg_score)
    fixed_model_names.append(m_fixed)

print(fixed_model_names)
print(avg_scores)

# Now get indices and model names of models where we could extract score
model_indices_with_score = [i for i, s in enumerate(avg_scores) if s is not None]
scored_models = [(i, model_names_from_outputs[i], avg_scores[i]) for i in model_indices_with_score]

# Sort by average score, descending
scored_models_sorted = sorted(scored_models, key=lambda x: x[2], reverse=True)


# 30% strongest

import math
n_strongest = math.ceil(0.3 * len(scored_models_sorted))
strongest_last_30_percent = scored_models_sorted[:n_strongest]

# As lists:
strongest_last_30_percent_model_names = [x[1] for x in strongest_last_30_percent]
strongest_last_30_percent_avg_scores = [x[2] for x in strongest_last_30_percent]

print("+==============================================+")
print("Last 30% strongest models (with scores):")
for name, score in zip(strongest_last_30_percent_model_names, strongest_last_30_percent_avg_scores):
    print(f"{name}: {score}")

pickle.dump(strongest_last_30_percent_model_names, open("data/mmlu_fields/strongest_last_30_percent_models.pkl", "wb"))


# 50% weakest
# 50% weakest models (lowest scores)

import math
n_weakest = math.ceil(0.5 * len(scored_models_sorted))
weakest_50_percent = scored_models_sorted[-n_weakest:]

# As lists:
weakest_50_percent_model_names = [x[1] for x in weakest_50_percent]
weakest_50_percent_avg_scores = [x[2] for x in weakest_50_percent]

print("+==============================================+")
print("Weakest 50% models (with scores):")
for name, score in zip(weakest_50_percent_model_names, weakest_50_percent_avg_scores):
    print(f"{name}: {score}")

pickle.dump(weakest_50_percent_model_names, open("data/mmlu_fields/weakest_50_percent_models.pkl", "wb"))



# Take the top 40
strongest_40 = scored_models_sorted[:40]
# Take the last 30% (round up if needed) of the sorted strong models

# As lists:
strongest_40_model_names = [x[1] for x in strongest_40]
strongest_40_avg_scores = [x[2] for x in strongest_40]

print("+==============================================+")
print("Strongest 40 models (with scores):")
for name, score in zip(strongest_40_model_names, strongest_40_avg_scores):
    print(f"{name}: {score}")

os.makedirs("data/mmlu_fields", exist_ok=True)
pickle.dump(strongest_40_model_names, open("data/mmlu_fields/strongest_models.pkl", "wb"))



Loaded model_outputs.pickle keys: dict_keys(['data', 'models'])
['abacusai/MetaMath-bagel-34b-v0.2-c1500', 'zhengr/MixTAO-7Bx2-MoE-DPO', 'alignment-handbook/zephyr-7b-sft-full', 'LoSboccacc/orthogonal-2x7B-base', 'rombodawg/Leaderboard-killer-MoE_4x7b', 'moreh/MoMo-70B-lora-1.8.6-DPO', 'FelixChao/ExtremeDolphin-MoE', 'rombodawg/Everyone-Coder-4x7b-Base', 'nfaheem/Marcoroni-7b-DPO-Merge', 'Swisslex/Mixtral-Orca-v0.1', 'deepseek-ai/deepseek-moe-16b-base', 'wang7776/Mistral-7B-Instruct-v0.2-sparsity-20', 'BAAI/Aquila2-34B', 'maywell/PiVoT-SUS-RP', 'HenryJJ/dolphin-2.6-mistral-7b-dpo-orca-v3', 'h2m/mhm-7b-v1.3', 'macadeliccc/polyglot-math-4x7b', 'Locutusque/Rhino-Mistral-7B', 'CallComply/Starling-LM-11B-alpha', 'macadeliccc/laser-dolphin-mixtral-2x7b-dpo', 'CallComply/SOLAR-10.7B-Instruct-v1.0-128k', 'bhavinjawade/SOLAR-10B-Nector-DPO-Jawade', 'CallComply/zephyr-7b-beta-128k', 'cloudyu/Yi-34Bx3-MoE-90B', 'Weyaxi/Bagel-Hermes-34B-Slerp', 'Weyaxi/Bagel-Hermes-2x34b', 'PSanni/MPOMixtral-8x7B-